In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

# Course difficulty = low pass rate + high submission delay + low avg score
# Source: silver_assessments joined with silver_students

assessments = spark.table("silver_assessments")
students = spark.table("silver_students")

# Avg score and submission delay per course
course_assessment_stats = assessments \
    .groupBy("course_id") \
    .agg(
        F.avg("score_filled").alias("avg_score"),
        F.avg("submission_delay_days").alias("avg_submission_delay"),
        F.count("*").alias("total_submissions"),
        F.sum(F.when(F.col("submission_status") == "not_submitted", 1).otherwise(0)).alias("not_submitted_count")
    )

# Pass rate per course from student outcomes
course_pass_stats = students \
    .groupBy("course_id") \
    .agg(
        F.count("*").alias("total_students"),
        F.sum(F.when(F.col("final_result").isin("Pass", "Distinction"), 1).otherwise(0)).alias("passed_students"),
        F.sum(F.when(F.col("final_result") == "Withdrawn", 1).otherwise(0)).alias("withdrawn_students"),
        F.sum(F.when(F.col("final_result") == "Fail", 1).otherwise(0)).alias("failed_students")
    ) \
    .withColumn("pass_rate", F.round(F.col("passed_students") / F.col("total_students"), 4)) \
    .withColumn("dropout_rate", F.round(F.col("withdrawn_students") / F.col("total_students"), 4)) \
    .withColumn("fail_rate", F.round(F.col("failed_students") / F.col("total_students"), 4))

# Join both
course_stats = course_pass_stats.join(course_assessment_stats, on="course_id", how="left")

# Difficulty score: higher when pass rate low, avg score low, delay high
# Normalized 0-100 scale
# difficulty = (1 - pass_rate)*40 + (1 - avg_score/100)*40 + min(avg_delay/30, 1)*20
course_stats = course_stats \
    .withColumn("avg_score_norm", F.col("avg_score") / 100) \
    .withColumn("delay_norm", F.least(F.col("avg_submission_delay") / 30, F.lit(1.0))) \
    .withColumn("difficulty_score", F.round(
        (1 - F.col("pass_rate")) * 40 +
        (1 - F.col("avg_score_norm")) * 40 +
        F.col("delay_norm") * 20,
        2
    )) \
    .withColumn("difficulty_band", F.when(F.col("difficulty_score") >= 60, "Hard")
                                    .when(F.col("difficulty_score") >= 40, "Medium")
                                    .otherwise("Easy")) \
    .withColumn("_computed_at", F.current_timestamp())

gold_course_difficulty = course_stats.select(
    "course_id", "total_students", "passed_students", "withdrawn_students",
    "failed_students", "pass_rate", "dropout_rate", "fail_rate",
    "avg_score", "avg_submission_delay", "total_submissions",
    "not_submitted_count", "difficulty_score", "difficulty_band", "_computed_at"
)

gold_course_difficulty.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_course_difficulty")

print(f"✅ gold_course_difficulty: {gold_course_difficulty.count()} rows")
gold_course_difficulty.orderBy("difficulty_score", ascending=False).show(truncate=False)

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 8, Finished, Available, Finished, False)

✅ gold_course_difficulty: 7 rows
+---------+--------------+---------------+------------------+---------------+---------+------------+---------+-----------------+--------------------+-----------------+-------------------+----------------+---------------+--------------------------+
|course_id|total_students|passed_students|withdrawn_students|failed_students|pass_rate|dropout_rate|fail_rate|avg_score        |avg_submission_delay|total_submissions|not_submitted_count|difficulty_score|difficulty_band|_computed_at              |
+---------+--------------+---------------+------------------+---------------+---------+------------+---------+-----------------+--------------------+-----------------+-------------------+----------------+---------------+--------------------------+
|CCC      |4249          |1628           |1883              |738            |0.3831   |0.4432      |0.1737   |73.21884899683211|0.9840234948604992  |18940            |11                 |36.04           |Easy           |202

In [2]:
# Risk score = weighted combination of:
# - final_result (40%) — already known for historical students
# - num_of_prev_attempts (20%) — more attempts = higher risk
# - total clicks in VLE (20%) — low engagement = higher risk  
# - avg assessment score (20%) — low scores = higher risk

students = spark.table("silver_students")
assessments = spark.table("silver_assessments")
vle = spark.table("silver_vle_activity")

# Per-student avg score
student_scores = assessments \
    .groupBy("student_id") \
    .agg(F.avg("score_filled").alias("avg_score"))

# Per-student total clicks
student_clicks = vle \
    .groupBy("student_id") \
    .agg(F.sum("sum_click").alias("total_clicks"))

# Join everything onto students
risk_base = students \
    .join(student_scores, on="student_id", how="left") \
    .join(student_clicks, on="student_id", how="left") \
    .fillna({"avg_score": 0.0, "total_clicks": 0})

# Normalize clicks: cap at 10000 for scoring
risk_base = risk_base \
    .withColumn("clicks_norm", F.least(F.col("total_clicks") / 10000, F.lit(1.0))) \
    .withColumn("score_norm", F.col("avg_score") / 100) \
    .withColumn("attempt_penalty", F.least(F.col("num_of_prev_attempts") / 5, F.lit(1.0)))

# Risk score 0-100: higher = more at risk
risk_base = risk_base \
    .withColumn("risk_score", F.round(
        F.col("is_at_risk") * 40 +
        F.col("attempt_penalty") * 20 +
        (1 - F.col("clicks_norm")) * 20 +
        (1 - F.col("score_norm")) * 20,
        2
    )) \
    .withColumn("risk_band", F.when(F.col("risk_score") >= 60, "High")
                              .when(F.col("risk_score") >= 40, "Medium")
                              .otherwise("Low")) \
    .withColumn("_computed_at", F.current_timestamp())

gold_student_risk = risk_base.select(
    "student_id", "course_id", "semester", "gender", "age_band",
    "highest_education", "num_of_prev_attempts", "studied_credits",
    "final_result", "is_at_risk", "avg_score", "total_clicks",
    "risk_score", "risk_band", "_computed_at"
)

gold_student_risk.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_student_risk")

print(f"✅ gold_student_risk: {gold_student_risk.count():,} rows")

# Quick sanity check
gold_student_risk.groupBy("risk_band").count().orderBy("count", ascending=False).show()

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 4, Finished, Available, Finished, False)

✅ gold_student_risk: 28,785 rows
+---------+-----+
|risk_band|count|
+---------+-----+
|     High|14228|
|      Low|13472|
|   Medium| 1085|
+---------+-----+



In [3]:
# Effectiveness = avg pass rate + avg student score + avg engagement
# across all students in courses taught by each instructor
# Source: silver_instructors (synthetic map) + gold_course_difficulty + gold_student_risk

instructors = spark.table("silver_instructors")
course_diff = spark.table("gold_course_difficulty")
student_risk = spark.table("gold_student_risk")

# Avg score and risk per course
course_student_stats = student_risk \
    .groupBy("course_id") \
    .agg(
        F.avg("avg_score").alias("avg_student_score"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.avg("total_clicks").alias("avg_engagement_clicks")
    )

# Join instructor → course → stats
instructor_stats = instructors \
    .join(course_diff.select("course_id", "pass_rate", "dropout_rate", "difficulty_score"),
          on="course_id", how="left") \
    .join(course_student_stats, on="course_id", how="left")

# Effectiveness score 0-100: higher = more effective
# High pass rate, high avg score, low risk = effective instructor
instructor_stats = instructor_stats \
    .withColumn("score_norm", F.col("avg_student_score") / 100) \
    .withColumn("effectiveness_score", F.round(
        F.col("pass_rate") * 40 +
        F.col("score_norm") * 40 +
        (1 - F.col("dropout_rate")) * 20,
        2
    )) \
    .withColumn("effectiveness_band", F.when(F.col("effectiveness_score") >= 70, "High")
                                       .when(F.col("effectiveness_score") >= 50, "Medium")
                                       .otherwise("Low")) \
    .withColumn("_computed_at", F.current_timestamp())

gold_instructor_effectiveness = instructor_stats.select(
    "instructor_id", "instructor_name", "department", "course_id",
    "pass_rate", "dropout_rate", "avg_student_score", "avg_engagement_clicks",
    "effectiveness_score", "effectiveness_band", "_computed_at"
)

gold_instructor_effectiveness.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_instructor_effectiveness")

print(f"✅ gold_instructor_effectiveness: {gold_instructor_effectiveness.count()} rows")
gold_instructor_effectiveness.orderBy("effectiveness_score", ascending=False).show(truncate=False)

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 5, Finished, Available, Finished, False)

✅ gold_instructor_effectiveness: 7 rows
+-------------+---------------+--------------+---------+---------+------------+------------------+---------------------+-------------------+------------------+--------------------------+
|instructor_id|instructor_name|department    |course_id|pass_rate|dropout_rate|avg_student_score |avg_engagement_clicks|effectiveness_score|effectiveness_band|_computed_at              |
+-------------+---------------+--------------+---------+---------+------------+------------------+---------------------+-------------------+------------------+--------------------------+
|INST_001     |Dr. A. Rahman  |Mathematics   |AAA      |0.7191   |0.1629      |64.82899533408863 |1762.2780898876404   |71.44              |High              |2026-04-16 21:13:37.673214|
|INST_007     |Dr. G. Smith   |Business      |GGG      |0.5891   |0.1175      |64.16149796763665 |521.7957860615884    |66.88              |Medium            |2026-04-16 21:13:37.673214|
|INST_005     |Dr. E. Ngu

In [4]:
# Engagement = recency + frequency + depth of interaction
# Source: silver_lms_events

events = spark.table("silver_lms_events")

# Weights per event type — deeper interactions score higher
event_weights = {
    "login":                1,
    "lecture_viewed":       3,
    "quiz_attempted":       4,
    "quiz_completed":       5,
    "assignment_submitted": 5,
    "attendance_marked":    2,
    "forum_post":           3,
    "feedback_submitted":   2,
}

weight_expr = F.create_map([F.lit(x) for pair in event_weights.items() for x in pair])

events_weighted = events \
    .withColumn("event_weight", weight_expr[F.col("event_type")]) \
    .withColumn("event_weight", F.coalesce(F.col("event_weight"), F.lit(1)))

# Per student per course engagement
engagement = events_weighted \
    .groupBy("student_id", "course_id") \
    .agg(
        F.count("*").alias("total_events"),
        F.sum("event_weight").alias("weighted_event_score"),
        F.sum("duration_sec").alias("total_duration_sec"),
        F.countDistinct("event_date").alias("active_days"),
        F.max("event_ts").alias("last_active_ts"),
        F.avg("score").alias("avg_event_score")
    ) \
    .withColumn("avg_duration_min", F.round(F.col("total_duration_sec") / 60, 2)) \
    .withColumn("engagement_score", F.round(
        F.least(F.col("weighted_event_score") / 50, F.lit(1.0)) * 60 +
        F.least(F.col("active_days") / 10, F.lit(1.0)) * 40,
        2
    )) \
    .withColumn("engagement_band", F.when(F.col("engagement_score") >= 70, "High")
                                    .when(F.col("engagement_score") >= 40, "Medium")
                                    .otherwise("Low")) \
    .withColumn("_computed_at", F.current_timestamp())

gold_engagement = engagement.select(
    "student_id", "course_id", "total_events", "weighted_event_score",
    "total_duration_sec", "avg_duration_min", "active_days",
    "last_active_ts", "avg_event_score", "engagement_score",
    "engagement_band", "_computed_at"
)

gold_engagement.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_engagement")

print(f"✅ gold_engagement: {gold_engagement.count()} rows")
gold_engagement.groupBy("engagement_band").count().show()

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 6, Finished, Available, Finished, False)

✅ gold_engagement: 394 rows
+---------------+-----+
|engagement_band|count|
+---------------+-----+
|            Low|  394|
+---------------+-----+



In [7]:
# Weekly activity trend per course from VLE clickstream
# Shows which weeks are high/low engagement — useful for dashboard trend line

vle = spark.table("silver_vle_activity")
students = spark.table("silver_students")

# OULAD dates are integers = days from course start
# Group into weeks (every 7 days)
weekly = vle \
    .withColumn("week_number", (F.col("date") / 7).cast(IntegerType())) \
    .groupBy("course_id", "semester", "week_number") \
    .agg(
        F.sum("sum_click").alias("total_clicks"),
        F.countDistinct("student_id").alias("active_students"),
        F.avg("sum_click").alias("avg_clicks_per_student")
    ) \
    .withColumn("_computed_at", F.current_timestamp())

gold_weekly_trends = weekly.filter(F.col("week_number") >= 0)

gold_weekly_trends.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_weekly_trends")

print(f"✅ gold_weekly_trends: {gold_weekly_trends.count():,} rows")

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 9, Finished, Available, Finished, False)

✅ gold_weekly_trends: 817 rows


In [8]:
gold_tables = {
    "gold_course_difficulty":      ("course_id",    7),
    "gold_student_risk":           ("student_id", 28000),
    "gold_instructor_effectiveness":("instructor_id", 7),
    "gold_engagement":             ("student_id",    1),
    "gold_weekly_trends":          ("week_number",  10),
}

print("--- Gold Validation ---")
for table, (key_col, min_rows) in gold_tables.items():
    df = spark.table(table)
    count = df.count()
    nulls = df.filter(F.col(key_col).isNull()).count()
    row_status = "✅" if count >= min_rows else "❌"
    null_status = "✅" if nulls == 0 else f"⚠️  {nulls} nulls"
    print(f"{row_status} {table}: {count:,} rows | key nulls: {null_status}")

print("\n--- KPI Sanity Checks ---")

# Pass rates should be between 0 and 1
diff = spark.table("gold_course_difficulty")
bad_pass = diff.filter((F.col("pass_rate") < 0) | (F.col("pass_rate") > 1)).count()
print(f"{'✅' if bad_pass == 0 else '❌'} Pass rates in valid range: {bad_pass} violations")

# Risk scores should be between 0 and 100
risk = spark.table("gold_student_risk")
bad_risk = risk.filter((F.col("risk_score") < 0) | (F.col("risk_score") > 100)).count()
print(f"{'✅' if bad_risk == 0 else '❌'} Risk scores in valid range: {bad_risk} violations")

# Effectiveness scores should be between 0 and 100
eff = spark.table("gold_instructor_effectiveness")
bad_eff = eff.filter((F.col("effectiveness_score") < 0) | (F.col("effectiveness_score") > 100)).count()
print(f"{'✅' if bad_eff == 0 else '❌'} Effectiveness scores in valid range: {bad_eff} violations")

StatementMeta(, 57887ee4-93d9-4282-91a1-e5a510e6e56d, 10, Finished, Available, Finished, False)

--- Gold Validation ---
✅ gold_course_difficulty: 7 rows | key nulls: ✅
✅ gold_student_risk: 28,785 rows | key nulls: ✅
✅ gold_instructor_effectiveness: 7 rows | key nulls: ✅
✅ gold_engagement: 394 rows | key nulls: ✅
✅ gold_weekly_trends: 817 rows | key nulls: ✅

--- KPI Sanity Checks ---
✅ Pass rates in valid range: 0 violations
✅ Risk scores in valid range: 0 violations
✅ Effectiveness scores in valid range: 0 violations
